# Eyeballing the `expected_words` baseline

Sample a board, show the clue the model gives, the own words it intends
the guesser to find, and every board word ranked against that clue. The
point is to judge clue *quality* by eye, which no aggregate statistic
does — and to sanity-check the embedding itself, since a clue can only be
as good as the similarities underneath it.

Two numbers are shown per word:

- **cos** — raw cosine similarity in the embedding space. This is the
  number to squint at when asking "is this embedding any good?"
- **z** — that cosine expressed in the clue's own distribution across the
  400-word board vocabulary. This is what the model actually thresholds
  on. For a fixed clue it's a monotonic transform of cos, so the *ranking*
  is identical; only the scale differs.

Rank order is what decides a turn: a guesser works down its own similarity
ranking and stops at its first mistake, so a clue is only as good as its
worst intended word's position.

Boards are built from `load_holdout_wordlist()` — the 150 words held out
of training — so these are the same kind of boards the frozen eval suite
uses. Requires `cache/similarity_tensor.npy` and `cache/clue_stats.npz`
(`scripts/data/build_similarity_tensor.py`, `scripts/data/build_clue_stats.py`).

In [ ]:
import sys
from pathlib import Path

# Notebook lives in scratch/, one level below the project root.
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "codenames").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import random

import numpy as np

from codenames.board import Board, Role, load_holdout_wordlist, load_wordlist
from codenames.clue_stats import ClueStats
from codenames.similarity import SimilarityTensor
from codenames.spymasters.base import TurnContext
from codenames.spymasters.registry import spymaster_spec

sims = SimilarityTensor.load()
stats = ClueStats.load()

SPYMASTER_CLS, SPYMASTER_KWARGS = spymaster_spec("expected_words")
spymaster = SPYMASTER_CLS(**SPYMASTER_KWARGS)
SPACE = SPYMASTER_KWARGS.get("space", "numberbatch")
print(f"{type(spymaster).__name__}  space={SPACE}  params={SPYMASTER_KWARGS}")

In [ ]:
ROLE_TAG = {Role.OWN: "own", Role.OPPONENT: "OPPONENT", Role.NEUTRAL: "neutral", Role.ASSASSIN: "ASSASSIN"}
# How many of each role a board holds, so the reveal arguments can be
# range-checked with a message that says what the ceiling actually is.
ROLE_TOTAL = {Role.OWN: 9, Role.OPPONENT: 8, Role.NEUTRAL: 7, Role.ASSASSIN: 1}


def scores_for(clue: str, words: list[str], space: str = SPACE) -> dict[str, tuple[float, float]]:
    """`{word: (cosine, z)}`. The z is the cosine re-expressed in this
    clue's own distribution over the whole board vocabulary -- the same
    quantity the spymaster scores on, so what's printed is literally what
    it saw. Monotonic in cosine for a fixed clue, so both columns rank
    identically; showing raw cosine alongside is what makes it possible to
    judge whether the embedding is any good."""
    ci = sims.clue_index[clue.lower()]
    si = stats.space_index(space)
    mu, sd = float(stats.mean[ci, si]), float(stats.std[ci, si])
    bi = sims.board_index
    out = {}
    for w in words:
        cos = float(sims.tensor[ci, bi[w.lower()], si])
        out[w] = (cos, (cos - mu) / sd)
    return out


def sample_board(seed: int | None = None, reveal_own: int = 0, reveal_opponent: int = 0,
                 reveal_neutral: int = 0, holdout_only: bool = True) -> Board:
    """A board, optionally part-played, with the revealed count controlled
    per role -- which side is ahead changes the position a lot, and a
    single total can't express that. The assassin is never revealed (a
    revealed assassin means the game is already over), and at least one own
    word is always left unrevealed, since asking for a clue with none left
    is meaningless."""
    wanted = {Role.OWN: reveal_own, Role.OPPONENT: reveal_opponent, Role.NEUTRAL: reveal_neutral}
    for role, n in wanted.items():
        ceiling = ROLE_TOTAL[role] - (1 if role is Role.OWN else 0)
        if not 0 <= n <= ceiling:
            raise ValueError(f"reveal_{role.value}={n} out of range: a board has "
                             f"{ROLE_TOTAL[role]} {role.value} words, so at most {ceiling} can be revealed")

    seed = random.randrange(1_000_000) if seed is None else seed
    vocab = load_holdout_wordlist() if holdout_only else load_wordlist()
    board = Board.generate(seed, vocabulary=vocab)
    rng = random.Random(seed ^ 0x5EED)
    for role, n in wanted.items():
        if n:
            for w in rng.sample(board.words_by_role(role), n):
                board.reveal(w)
    return board

In [ ]:
def inspect(seed: int | None = None, reveal_own: int = 0, reveal_opponent: int = 0,
            reveal_neutral: int = 0, alternatives: int = 3, holdout_only: bool = True) -> None:
    """Print one board's clue plus every word ranked against it."""
    board = sample_board(seed, reveal_own, reveal_opponent, reveal_neutral, holdout_only)
    ctx = TurnContext(board=board, turn_index=len(board.revealed))
    picks = spymaster.top_clues(ctx, sims, alternatives)
    clue, number, score = picks[0]

    all_words = list(board.words)
    sc = scores_for(clue, all_words)
    unrevealed = [w for w in all_words if not board.is_revealed(w)]
    live = sorted(unrevealed, key=lambda w: -sc[w][1])
    by_role = {r: [w for w in live if board.role_of(w) is r] for r in ROLE_TAG}

    intended = by_role[Role.OWN][:number]           # the model intends the top-k own words
    weakest = sc[intended[-1]][1] if intended else float("nan")
    non_own = [w for w in live if board.role_of(w) is not Role.OWN]
    nearest = max(non_own, key=lambda w: sc[w][1]) if non_own else None

    revealed_note = ""
    if board.revealed:
        counts = {r: sum(1 for w in all_words if board.is_revealed(w) and board.role_of(w) is r) for r in ROLE_TAG}
        revealed_note = "  revealed: " + ", ".join(f"{n} {ROLE_TAG[r]}" for r, n in counts.items() if n)
    print(f"board {board.seed}{revealed_note}")
    print(f"CLUE: {clue.upper()}  {number}      score={score:.3f}      space={SPACE}")

    print(f"\n  intends ({number}):")
    for w in intended:
        cos, z = sc[w]
        print(f"    {w:<16s} cos={cos:+.3f}  z={z:+.2f}")

    print("\n  nearest word of each other role:")
    for role in (Role.OPPONENT, Role.NEUTRAL, Role.ASSASSIN):
        words = by_role[role]
        if not words:
            print(f"    {ROLE_TAG[role]:<9s} -- none left --")
            continue
        cos, z = sc[words[0]]
        print(f"    {ROLE_TAG[role]:<9s} {words[0]:<16s} cos={cos:+.3f}  z={z:+.2f}   "
              f"margin below weakest intended: {weakest - z:+.2f}")

    if nearest is not None:
        verdict = "CLEAR" if sc[nearest][1] < weakest else "*** OUTRANKED ***"
        print(f"\n  weakest intended {weakest:+.2f}  vs  nearest distractor {sc[nearest][1]:+.2f}   -> {verdict}")

    print(f"\n  every word, ranked by similarity to {clue.upper()}:")
    print(f"    {'':>3}  {'word':<16s} {'role':<9s} {'cos':>7s} {'z':>7s}")
    for i, w in enumerate(live, start=1):
        cos, z = sc[w]
        mark = "  <-- intended" if w in intended else ""
        print(f"    {i:>3}. {w:<16s} {ROLE_TAG[board.role_of(w)]:<9s} {cos:>+7.3f} {z:>+7.2f}{mark}")
    for w in sorted((x for x in all_words if board.is_revealed(x)), key=lambda x: -sc[x][1]):
        cos, z = sc[w]
        print(f"      -  {w:<16s} {ROLE_TAG[board.role_of(w)]:<9s} {cos:>+7.3f} {z:>+7.2f}  (revealed)")

    if alternatives > 1:
        print("\n  runners-up:")
        for c, n, s in picks[1:]:
            print(f"    {c:<16s} {n}   score={s:.3f}")
    print()

In [ ]:
inspect(seed=7)

In [ ]:
# Behind: they've taken 5, we've taken 2.
inspect(seed=7, reveal_own=2, reveal_opponent=5, reveal_neutral=1, alternatives=1)

In [ ]:
# Nearly finished: one own word left to clue for.
inspect(seed=7, reveal_own=8, reveal_opponent=3, reveal_neutral=4, alternatives=1)

In [ ]:
def sweep(n: int = 25, reveal_own: int = 0, reveal_opponent: int = 0, reveal_neutral: int = 0) -> None:
    """One line per board -- for skimming many clues quickly. `OUTRANKED`
    flags any board where a non-own word beats the weakest intended word,
    which is the failure this whole metric exists to avoid."""
    print(f"{'seed':>6} {'clue':<16} {'n':>2}  {'weakest':>8} {'nearest':>8} {'margin':>7}  worst distractor")
    bad = 0
    for seed in range(n):
        board = sample_board(seed, reveal_own, reveal_opponent, reveal_neutral)
        ctx = TurnContext(board=board, turn_index=len(board.revealed))
        clue, number, _ = spymaster.top_clues(ctx, sims, 1)[0]
        live = [w for w in board.words if not board.is_revealed(w)]
        sc = scores_for(clue, live)
        own = sorted((w for w in live if board.role_of(w) is Role.OWN), key=lambda w: -sc[w][1])
        non_own = [w for w in live if board.role_of(w) is not Role.OWN]
        weakest = sc[own[number - 1]][1]
        nearest = max(non_own, key=lambda w: sc[w][1])
        flag = "  <-- OUTRANKED" if sc[nearest][1] >= weakest else ""
        bad += bool(flag)
        print(f"{board.seed:>6} {clue:<16} {number:>2}  {weakest:>+8.2f} {sc[nearest][1]:>+8.2f} "
              f"{weakest - sc[nearest][1]:>+7.2f}  {nearest} ({ROLE_TAG[board.role_of(nearest)]}){flag}")
    print(f"\nboards where a non-own word outranks the weakest intended word: {bad}/{n}")


sweep(15)